# Opacity Study (update: Mar 15 by Shano)

In this example, we'll create a full two-alternative forced choice (2AFC) study for visualizations of correlated data (from [1]), using vega components. We'll use numpy and scipy to generate a dataset, pandas to create a well-structured data frame, and the `revisitpy` package to tie it all together.

You'll see that we have a section which uses the `revisitpy` Widget. This widget is a fully interactive jupyter notebook widget which allows us to preview the created study directly in the notebook. You'll notice that we also utilize the `revisitpy_server` package. This is to simplify the process of viewing our widget. Specifically, it removes the requirement of having a version of the reVISit repository running locally on your computer.

[1] Harrison, Lane, Fumeng Yang, Steven Franconeri, and Remco Chang. "Ranking visualizations of correlation using weber's law." IEEE transactions on visualization and computer graphics 20, no. 12 (2014): 1943-1952.

We'll start by defining the basic structure of the study:

In [113]:
import revisitpy as rvt
import numpy as np
import pandas as pd
import altair as alt
import vl_convert as vlc
import itertools
import revisitpy_server as rs
import json
import time
import anywidget
import vega



# Meta Data
study_metadata = rvt.studyMetadata(
    authors=["Shano Liang"],
    organizations=["VIS Lab"],
    title='Opacity Judgment Study',
    description='',
    date='2025-03-06',
    version='1.0'
)


# UI Config
ui_config = rvt.uiConfig(
  contactEmail="sliang1@wpi.edu",
  logoPath="./assets/revisitLogoSquare.svg",
  sidebar=True,
  withProgressBar=False,
  nextOnEnter=True
)

# Introduction
introduction = rvt.component(type='markdown', path='./assets/introduction.md', component_name__= 'introduction')

# Snippet of the introduction component.
print(introduction)

{
    "path": "./assets/introduction.md",
    "response": [],
    "type": "markdown"
}


# Generate X_segments for curves' generation (March 23)

In [114]:
def generate_random_x_segments(x_start=0.0, x_end=1.0, num_segments=3, seed=None):
    """
    randomly generate x_segments，with different step(s)
    """
    if seed is not None:
        np.random.seed(seed + 999)  # To avoid conflicts with seeds

    segment_points = np.sort(np.random.uniform(x_start, x_end, num_segments - 1))
    segment_points = np.concatenate([[x_start], segment_points, [x_end]])

    steps = np.random.choice([0.005, 0.01, 0.02, 0.04, 0.05], size=num_segments)
    segments = [(float(segment_points[i]), float(segment_points[i + 1]), float(steps[i]))
                for i in range(num_segments)]

    return segments

## Generate Curves

In [115]:
def generate_smooth_curve(seed=None, wave_combinations=None, x_segments=None):
    """
    Generate a smooth random curved line with flexible x spacing.
    """
    if seed is not None:
        np.random.seed(seed)

    freq_factors = np.random.randint(1, 3, size=4).tolist()
    amp_factors = np.random.uniform(0.5, 10, size=4).tolist()
    noise_level = np.random.uniform(0, 0.02)
    x_shift = np.random.uniform(0, 3)
    y_shift = np.random.uniform(0, 3)

    if wave_combinations is None:
        wave_combinations = [['sin', 'sinc']]

    # Default segments: equivalent to linspace(0, 10, 100)
    if x_segments is None:
        x_segments = generate_random_x_segments(x_start=0.0, x_end=1.0, num_segments=3, seed=seed)

    x_parts = [np.arange(start, end, step) for start, end, step in x_segments]
    x = np.concatenate(x_parts)

    y = np.zeros_like(x)

    for a, f, waves in zip(amp_factors, freq_factors, wave_combinations):
        for w in waves:
            if w == 'sin':
                y += a * np.sin(f * (x + x_shift))
            elif w == 'cos':
                y += a * np.cos(f * (x + x_shift))
            elif w == 'sinc':
                y += a * np.sinc(f * (x + x_shift))
            elif w == 'tanh':
                y += a * np.tanh(f * (x + x_shift))
            elif w == 'exp':
                y += a * np.exp(-0.5 * f * (x + x_shift))

    y += (np.random.normal(scale=noise_level, size=len(x)) + y_shift)
    return x, y

## Plot Altair Vis

We now want to generate the datasets that will go into our vega charts. We don't yet have to worry about rendering these, we'll just define the functions to generate the data.

In [116]:
def plot_altair_curve(seed=None, num_curves=3, opacity=0.3, x_segments=None):
    """
    Generate and plot multiple smooth random curved lines using Altair with shaded areas.
    """
    if seed is not None:
        np.random.seed(seed)  # Set seed for reproducibility
    
    curves_data = []
    shaded_data = []
    y_min_global = float('inf')
    y_max_global = float('-inf')

    if x_segments is None:
        x_segments_preview = generate_random_x_segments(seed=seed)
        print("Auto-generated x_segments:", x_segments_preview)
    else:
        x_segments_preview = x_segments
        print("Using custom x_segments:", x_segments_preview)

    for i in range(num_curves):
        curve_seed = seed + i if seed is not None else None
        x, y = generate_smooth_curve(seed=curve_seed, x_segments=x_segments_preview)
        df = pd.DataFrame({'X': x, 'Y': y, 'Curve': f'Curve {i+1}'})
        curves_data.append(df)
        y_min_global = min(y_min_global, np.min(y))
        y_max_global = max(y_max_global, np.max(y))
    
    for df in curves_data:
        df_shade = df.copy()
        df_shade['Y0'] = y_min_global  # Shade from the global minimum y-value up to the curve
        shaded_data.append(df_shade)
    
    all_curves = pd.concat(curves_data)
    all_shaded = pd.concat(shaded_data)
    
    y_scale = alt.Scale(domain=[y_min_global, y_max_global])  # Auto-scale y-axis
    
    line_chart = alt.Chart(all_curves).mark_line().encode(
        x='X:Q',
        y=alt.Y('Y:Q', scale=y_scale),
        color=alt.Color('Curve:N', legend=alt.Legend(title="Curves"))
    )
    
    shaded_chart = alt.Chart(all_shaded).mark_area(opacity=opacity).encode(
        x='X:Q',
        y=alt.Y('Y0:Q', scale=y_scale),
        y2='Y:Q',
        color=alt.Color('Curve:N', legend=None)  # Use the same color as the line but without an extra legend
    )
    
    return (shaded_chart + line_chart).properties(
        width=400,
        height=300,
        title="Curves"
    )

# Use the function to print a test plot
plot_altair_curve(seed=None, num_curves=3, opacity=0.3, x_segments=None)

Auto-generated x_segments: [(0.0, 0.05889892137207742, 0.02), (0.05889892137207742, 0.7110956862030287, 0.04), (0.7110956862030287, 1.0, 0.01)]


alt.LayerChart(...)

# Side by Side

For this study, we need to generate pairs of scatterplots and pairs of parallel coordinate plots. We will create two generalized functions which take in two data frames whose columns are 'X' and 'Y' and whose entries are tuples, indicating the coordinates. These functions will each return a vega-altair chart that will be added as components.

In [117]:
def plot_side_by_side(seed=None, num_curves=3, opacityGroup=[0.3, 0.6], x_segments=None):
    """
    Generate and display two Altair charts side by side with different opacities,
    using shared or generated x_segments.
    """
    if x_segments is None:
        x_segments = generate_random_x_segments(seed=seed)
        print("Auto-generated x_segments for side-by-side:", x_segments)
    else:
        print("Using provided x_segments for side-by-side:", x_segments)

    chart1 = plot_altair_curve(seed=seed, num_curves=num_curves, opacity=opacityGroup[0], x_segments=x_segments)
    chart2 = plot_altair_curve(seed=seed, num_curves=num_curves, opacity=opacityGroup[1], x_segments=x_segments)

    return alt.hconcat(chart1, chart2)

# x_segments = [(0, 0.5, 0.01), (0.5, 0.8, 0.02), (0.8, 1.0, 0.04)]
# chart = plot_side_by_side(seed=42, num_curves=3, opacityGroup=[0.3, 0.6], x_segments=x_segments)
# chart

chart = plot_side_by_side(seed=42, num_curves=3, opacityGroup=[0.3, 0.6])
chart

Auto-generated x_segments for side-by-side: [(0.0, 0.021672243636328248, 0.05), (0.021672243636328248, 0.5639069501395994, 0.04), (0.5639069501395994, 1.0, 0.05)]
Using custom x_segments: [(0.0, 0.021672243636328248, 0.05), (0.021672243636328248, 0.5639069501395994, 0.04), (0.5639069501395994, 1.0, 0.05)]
Using custom x_segments: [(0.0, 0.021672243636328248, 0.05), (0.021672243636328248, 0.5639069501395994, 0.04), (0.5639069501395994, 1.0, 0.05)]


alt.HConcatChart(...)

# Generate Vega Spec to combine Generated Data and Plots

Now that we have our functions to create the individual chart, we want a function that returns the correct vega spec when given the number of points, the correlation values, and the visualization type ('scatterPlot' or 'parallelPlot'). We'll use the number of points and the pair of correlation values to generate the dataset. Using the visualization type, we'll either return the scatter plot of this data or the parallel coordinates plot. Hover, instead of returning the vega-altair chart, we instead convert the chart to its vega-lite specification, then convert that into the true vega specification.


In [118]:
def create_vega_spec(visType, seed, num_curves=3, opacityGroup=[0.3,0.6], x_segments=None):
    """
    Generate a Vega spec from the Altair chart by converting it to Vega-Lite and then to Vega.
    """
    if visType == 'altairPlot':
        chart = plot_side_by_side(seed=seed, num_curves=num_curves, opacityGroup = opacityGroup, x_segments=x_segments)
    else:
        raise ValueError("Unsupported visualization type. Use 'altairPlot'.")
    
    vega_lite_spec = chart.to_json()
    vega_spec = vlc.vegalite_to_vega(vega_lite_spec, vl_version="5.20")
    return vega_spec

# We can print the test vega specification above to inspect its contents.
my_vega_spec = create_vega_spec(visType='altairPlot', seed=42, num_curves=3, opacityGroup=[0.3, 0.6])
#print(my_vega_spec)
json_spec = json.dumps(my_vega_spec, indent=2)
print(json_spec)

# my_segments = [(0, 0.4, 0.01), (0.4, 0.7, 0.02), (0.7, 1.0, 0.04)]
vega = create_vega_spec(
    visType='altairPlot',
    seed=42,
    num_curves=3,
    # x_segments=my_segments,
    opacityGroup=[0.3, 0.6]
)


Auto-generated x_segments for side-by-side: [(0.0, 0.021672243636328248, 0.05), (0.021672243636328248, 0.5639069501395994, 0.04), (0.5639069501395994, 1.0, 0.05)]
Using custom x_segments: [(0.0, 0.021672243636328248, 0.05), (0.021672243636328248, 0.5639069501395994, 0.04), (0.5639069501395994, 1.0, 0.05)]
Using custom x_segments: [(0.0, 0.021672243636328248, 0.05), (0.021672243636328248, 0.5639069501395994, 0.04), (0.5639069501395994, 1.0, 0.05)]
{
  "$schema": "https://vega.github.io/schema/vega/v5.json",
  "background": "white",
  "padding": 5,
  "height": 300,
  "data": [
    {
      "name": "data-d91bee8df115893b5576f29c12353b7b",
      "format": {},
      "values": [
        {
          "Curve": "Curve 1",
          "X": 0,
          "Y": 6.524268345675902,
          "Y0": -1.8952523072427736
        },
        {
          "Curve": "Curve 1",
          "X": 0.021672243636328248,
          "Y": 6.357179314447562,
          "Y0": -1.8952523072427736
        },
        {
          "C

# Creating The Component Function & Interaction Signals for ReVISit Trials

The `component_function` is used to transform every component in a given sequence to any new component. If we have a sequence that is the correct _structure_, then we call the `component()` method on that sequence and pass in the desired `component_function`. Any `meta` attributes in the original components are passed in as arguments to the `component_function`. 

We'll create a component function which takes in the visualization type, the correlation values, and the number of points and returns the correct vega specification component. Additionally, we append signals directly into the vega spec so that we can detect the user's right and left arrow keys. Instead of the user having to choose "left" or "right" in some drop down, the user will be able to use the left and right arrow keys to pick the chart. We add an additional signal to "highlight" the selected chart with a thick blue border. Finally, since we specified "nextOnEnter" as "True" in the "uiConfig", the user will also be able to proceed to the next component by pressing the "Enter" key. All of this combined creates a seamless study experience.

In [120]:
def component_function(seed=None, opacityGroup=None):
    if seed is not None and opacityGroup is not None:
        vega_spec = create_vega_spec(visType='altairPlot', seed=seed, num_curves=3, opacityGroup=opacityGroup)

        # Add signals to vega_spec for reactive response
        if 'config' not in vega_spec:
            vega_spec['config'] = {}

        vega_spec["config"]["signals"] = [
            {
                "name": "revisitAnswer",
                "value": {},
                "on": [
                    {
                        "events": "@concat_0_group:click",
                        "update": "{responseId: 'vegaDemoResponse1', response: 'left'}"
                    },
                    {
                        "events": "@concat_1_group:click",
                        "update": "{responseId: 'vegaDemoResponse1', response: 'right'}"
                    },
                    {
                        "events": {"source": "window", "type": "keydown"},
                        "update": "event.key === 'ArrowLeft' ? {responseId: 'vegaDemoResponse1', response: 'left'} : event.key === 'ArrowRight' ? {responseId: 'vegaDemoResponse1', response: 'right'} : revisitAnswer"
                    }
                ]
            }
        ]

        # Add visual stroke highlight based on selection
        for entry in vega_spec['marks']:
            if entry['name'] == 'concat_0_group':
                condition = 'left'
            elif entry['name'] == 'concat_1_group':
                condition = 'right'
            else:
                continue

            entry['encode']['update']['stroke'] = {
                "signal": f"revisitAnswer.response === '{condition}' ? 'blue' : null"
            }
            entry['encode']['update']['strokeWidth'] = {
                "signal": f"revisitAnswer.response === '{condition}' ? 3 : 0"
            }

        return rvt.component(
            type='vega',
            config=vega_spec,
            component_name__=f'{seed}-{opacityGroup[0]},{opacityGroup[1]}',
            response=[
                rvt.response(
                    id='vegaDemoResponse1',
                    prompt='You Selected:',
                    location='sidebar',
                    type='reactive',
                    required=True
                )
            ]
        )
    

    return rvt.component(
        type='questionnaire',
        component_name__='blank-component'
    )

# You can print the output of our component function with some test values.
comp_func = component_function(seed=42, opacityGroup=[0.3,0.6])

print(comp_func)

#print(component_function(seed=42, opacityGroup=[0.3, 0.6]))


Auto-generated x_segments for side-by-side: [(0.0, 0.021672243636328248, 0.05), (0.021672243636328248, 0.5639069501395994, 0.04), (0.5639069501395994, 1.0, 0.05)]
Using custom x_segments: [(0.0, 0.021672243636328248, 0.05), (0.021672243636328248, 0.5639069501395994, 0.04), (0.5639069501395994, 1.0, 0.05)]
Using custom x_segments: [(0.0, 0.021672243636328248, 0.05), (0.021672243636328248, 0.5639069501395994, 0.04), (0.5639069501395994, 1.0, 0.05)]
{
    "config": {
        "$schema": "https://vega.github.io/schema/vega/v5.json",
        "background": "white",
        "padding": 5,
        "height": 300,
        "data": [
            {
                "name": "data-d91bee8df115893b5576f29c12353b7b",
                "format": {},
                "values": [
                    {
                        "Curve": "Curve 1",
                        "X": 0,
                        "Y": 6.524268345675902,
                        "Y0": -1.8952523072427736
                    },
                

# Permuting the Final Sequence 

Here we generate the different combinations of the correlation values that we'd like (every combination of two numbers between 0 and 1 with precision 1). Then, we generate a fixed order sequence and being the permutations over our factors. We first permute over the visualization type, then over the number of points, then over all possible correlation value pairs.

When we permute over these factors, the corresponding factored will be added to the `meta` attributes of each component. By the end of these three permutations, we will have components that have 'visType', 'numPoints', and 'corrValues' key-value pairs in their `meta` attribute. Before calling the `component` method, these are all "filler" or "placeholder" components with no real value aside from their metadata attributes. Once we call the `component` method, each component is passed through the inputted `component_function` which will take the existing metadata as arguments. Thus, by the end of this method chaining, each component will be the correct vega component.

After we have finished generating the sequence, we add the entire component block to an a sequence only containing the introduction.

In [95]:

# Generate all combinations of two values between 1 and 10
combinations = itertools.combinations(range(1, 11), 2)

# Create the dataset with values divided by 10
dataSet = [{'opacityGroup': [x / 10, y / 10]} for x, y in combinations]

main_sequence = rvt.sequence(order='fixed')

main_sequence.permute(
        factors=[{'seed': 42}],
        order='latinSquare',
    ).permute(
        factors=dataSet,
        order='random',
    ).component(component_function)
    
sequence = rvt.sequence(order='fixed',components=[introduction]) + main_sequence

study = rvt.studyConfig(
    schema="https://raw.githubusercontent.com/revisit-studies/study/v2.0.0-rc1/src/parser/StudyConfigSchema.json",
    uiConfig=ui_config,
    studyMetadata=study_metadata,
    sequence=sequence
)

# Prints the entire configuration file which is approximately 150,000 lines of JSON
print(study)


Auto-generated x_segments for side-by-side: [(0.0, 0.021672243636328248, 0.05), (0.021672243636328248, 0.5639069501395994, 0.04), (0.5639069501395994, 1.0, 0.05)]
Using custom x_segments: [(0.0, 0.021672243636328248, 0.05), (0.021672243636328248, 0.5639069501395994, 0.04), (0.5639069501395994, 1.0, 0.05)]
Using custom x_segments: [(0.0, 0.021672243636328248, 0.05), (0.021672243636328248, 0.5639069501395994, 0.04), (0.5639069501395994, 1.0, 0.05)]
Auto-generated x_segments for side-by-side: [(0.0, 0.021672243636328248, 0.05), (0.021672243636328248, 0.5639069501395994, 0.04), (0.5639069501395994, 1.0, 0.05)]
Using custom x_segments: [(0.0, 0.021672243636328248, 0.05), (0.021672243636328248, 0.5639069501395994, 0.04), (0.5639069501395994, 1.0, 0.05)]
Using custom x_segments: [(0.0, 0.021672243636328248, 0.05), (0.021672243636328248, 0.5639069501395994, 0.04), (0.5639069501395994, 1.0, 0.05)]
Auto-generated x_segments for side-by-side: [(0.0, 0.021672243636328248, 0.05), (0.021672243636328

In [84]:

# turn study object into JSON and save
study_json = study.__str__()  # get JSON string
study_data = json.loads(study_json)  # turn into Python dictionary
#study_data["studyMetadata"]["title"] = "Opacity Judgment Study"

# save as config.json
with open("config.json", "w", encoding="utf-8") as f:
    json.dump(study_data, f, indent=2)

print("✅ config.json generated!")

✅ config.json generated!


# Using `revisitpy_server` to Prepare Our Widget

The `revisitpy` package provides a widget in order to preview our study directly in a Jupyter notebook. We can interact with the study, check that vega signals work, and even create some introductory data ourselves. In order for the widget to work, a local copy of the reVISit must be running on your local computer. If you already have reVISit locally (colloqioully our `study` repo), then all you need to do is navigate to your repository and run `yarn serve`. After this, the widget we create in this jupyter notebook will be useable.

A simpler way to achieve the same goal, however is using the `revisitpy_server` Python package. This is a simple python package which already has the most recent reVISit repository built and runs a server locally. After installing `revisitpy_server`, all that is required is the following:

In [96]:
process = rs.serve()

Server is running in the background at http://localhost:8080


# The Widget

Now that your server is running, we create the widget with the configuration file we created above. When calling the widget, we are assuming that the assets referenced in the configuration file are relative to this notebook. The widget then copies these static assets to the appropriate directory. Since we're currently using the `revisitpy_server` package, you'll see that they copied into the assets of the local virtual environment `revisitpy_server` package.

In [97]:
w = rvt.widget(study, server=True)

# In your own Jupyter notebook, calling `w` will now display the widget in a fully interactive manner.
w

Copying file from ./assets/introduction.md to d:\revisit\revisitpy-examples-main\revisitpy-examples-main\.venv\Lib\site-packages\revisitpy_server/static/__revisit-widget/assets/introduction.md
Copying file from ./assets/revisitLogoSquare.svg to d:\revisit\revisitpy-examples-main\revisitpy-examples-main\.venv\Lib\site-packages\revisitpy_server/static/__revisit-widget/assets/revisitLogoSquare.svg


Widget(config={'$schema': 'https://raw.githubusercontent.com/revisit-studies/study/v2.0.0-rc1/src/parser/Study…

# Optional: Data Collection

Now that we have the widget running, we can check out some sample data that would be generated from a user. Start by going through a small portion of the study. Once you've gone through the desired number of components inside the widget, navigate to the analysis dashboard using the 'Analysis' tab in the upper left-hand corner. Here you'll see individual participants and the data that they've generated. 

From here, we can export this data back into our Jupyter notebook. Start by clicking the "Download as Tidy CSV" on the right-hand side above the table. Here you'll be shown a preview of the CSV file with some additional options to truncate the data. In the bottom right-hand corner, you'll see a button with the Python icon. Clicking on this button will send the Tidy CSV back to the Jupyter notebook. Once the button is clicked, we can preview the data like so:

In [69]:
w.get_df()

KeyError: 'rows'

# Optional: Terminate the server

Closing the notebook will automatically terminate the server. If you'd rather do this manually, you can do the following.

In [70]:
process.terminate()